# Sampler comparison: Metropolis vs Gibbs

Reads the output of `evaluate/compare_samplers.py` (folders in `evaluate/output/`), which is run separately from the project root. It does not use the training runs of `results/`; those are analysed in `analysis.ipynb`.

```bash
uv run python evaluate/compare_samplers.py --N 4 20
```

For every N the script:

1. **Tunes the `step_size` of both samplers automatically** so that their acceptance falls in `[ACC_MIN, ACC_MAX]` of `train.py`. It tunes once at the initial θ and again at a reference θ trained with Metropolis, because the acceptance drops while ψ narrows.
2. **Trains from the same initial θ, with the same key**, once with each sampler, using the `n_sweep` of the config. Only the sampler changes.
3. **Samples the reference θ with both samplers recording every step** (`n_sweep = 1`). This makes the autocorrelation time independent of how much each sampler thins its chain:
   - **agreement:** both samplers draw from $|\psi|^2$, so $\langle\sum_i|x_i|^2\rangle$ and $\langle E_{loc}\rangle$ must agree within the error bars (errors from the spread of independent chain means);
   - **efficiency:** $\tau$ is the integrated autocorrelation time **per step** of $\sum_i|x_i|^2$, and the effective samples per second are $\text{ESS/s} = n_{steps}\,n_{chains} / (\tau\, t)$. This is the fair figure of merit: it accounts for both the cost of a step and how correlated consecutive steps are.

`COMPARE_DIR = None` loads the most recent comparison in `evaluate/output/`. Run it from VS Code with the `.venv` kernel.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

#The notebook lives in notebooks/, but evaluate/output/ is in the project root (where pyproject.toml is).
#This notebook only reads json and npz files, so it does not need JAX nor the project modules
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)

## 0. Load a comparison

Tuned step sizes per N and sampler, then the summary table of the training (test 3) and of the fixed reference θ (tests 1 and 2).

In [ ]:
import json

COMPARE_DIR = None   #e.g. ROOT / "evaluate/output/compare_N4-20_20261009-110805"; None = most recent
SAMPLER_NAMES = ["metropolis", "gibbs"]
SAMPLER_COLORS = {"metropolis": "tab:blue", "gibbs": "tab:orange"}

#Sorted by the date-time at the end of the name (sorting by the whole name would order by the list of N)
compare_dirs = sorted((ROOT / "evaluate" / "output").glob("compare_N*"), key=lambda d: d.name.rsplit("_", 1)[-1])
if COMPARE_DIR is None:
    if not compare_dirs:
        raise FileNotFoundError("No comparison found: run  uv run python evaluate/compare_samplers.py  first")
    COMPARE_DIR = compare_dirs[-1]
COMPARE_DIR = Path(COMPARE_DIR)

with open(COMPARE_DIR / "summary.json") as f:
    cmp = json.load(f)
cmp_hist = np.load(COMPARE_DIR / "histories.npz")
Ns = sorted(int(N) for N in cmp["N"])
res = {N: cmp["N"][str(N)] for N in Ns}
a = cmp["args"]

print(f"Comparison: {COMPARE_DIR.name}   ({len(compare_dirs)} available)")
print(f"N = {Ns}, {a['n_iter']} SR iterations per training, {a['n_steps']} steps at fixed θ, "
      f"E_loc every {a['e_stride']} steps, acceptance range {cmp['acc_range']}\n")

#Step sizes found by the automatic tuning, at the initial theta and at the reference theta
print(f"{'N':>4}  {'sampler':<11}{'step (init θ)':>15}{'acc':>6}{'step (ref θ)':>15}{'acc':>6}{'trials':>8}")
for N in Ns:
    for name in SAMPLER_NAMES:
        t0, t1 = res[N]["tuning_init"][name], res[N]["tuning_ref"][name]
        print(f"{N:>4}  {name:<11}{t0['step_size']:>15.4f}{t0['acceptance']:>6.2f}{t1['step_size']:>15.4f}"
              f"{t1['acceptance']:>6.2f}{len(t0['trials']) + len(t1['trials']):>8}")

In [ ]:
#Summary per N and sampler: training (test 3) and fixed reference theta (tests 1 and 2)
header = (f"{'N':>4}  {'sampler':<11}{'E_final':>20}{'acc':>6}{'ms/iter':>9}{'it<tol':>8}{'s<tol':>8}"
          f"{'μs/step':>10}{'τ r2':>8}{'ESS/s r2':>11}{'τ E':>7}")
print(header)
print("-" * len(header))
for N in Ns:
    tr, fx = res[N]["training"], res[N]["fixed"]
    for name in SAMPLER_NAMES:
        t, f = tr[name], fx[name]
        E = f"{t['E_final']:.5f}±{t['E_err']:.5f}"
        s_tol = f"{t['time_tol']:.1f}" if t["time_tol"] is not None else "-"
        print(f"{N:>4}  {name:<11}{E:>20}{t['acc_final']:>6.2f}{1e3 * t['time_per_iter']:>9.1f}{str(t['iter_tol']):>8}"
              f"{s_tol:>8}{1e6 * f['time_per_step']:>10.1f}{f['r2']['tau']:>8.2f}{f['r2']['ess_per_s']:>11.3g}"
              f"{f['E']['tau']:>7.2f}")
    ag = fx["agreement"]
    gain = fx["gibbs"]["r2"]["ess_per_s"] / fx["metropolis"]["r2"]["ess_per_s"]
    print(f"{'':>6}agreement M−G: Σ|x|² {ag['r2']:+.2f} σ, E {ag['E']:+.2f} σ   |   ESS/s Gibbs / Metropolis = {gain:.2f}")
print(f"\nτ E is in units of {a['e_stride']} steps. it<tol, s<tol: iterations and seconds (without compilation) "
      f"to reach a relative error below {a['rel_tol']:g}")

## 1. Training with each sampler

Same initial θ and same key, so the only difference is the sampler. Left: relative error per SR iteration, which measures the quality of the samples SR receives. Centre: the same against wall time (compilation excluded), which adds the cost of each sampler. Right: acceptance during training. Gibbs acceptance is per particle.

In [ ]:
fig, axes = plt.subplots(len(Ns), 3, figsize=(16, 3.6 * len(Ns)), squeeze=False)
for row, N in zip(axes, Ns):
    ax_it, ax_t, ax_acc = row
    for name in SAMPLER_NAMES:
        h = {field: cmp_hist[f"N{N}_{name}_{field}"] for field in ["rel_err", "cum_time", "acceptance"]}
        it = np.arange(1, len(h["rel_err"]) + 1)
        c = SAMPLER_COLORS[name]
        ax_it.semilogy(it, h["rel_err"], color=c, label=name)
        ax_t.semilogy(h["cum_time"], h["rel_err"], color=c, label=name)
        ax_acc.plot(it, h["acceptance"], color=c, label=name)
    for ax in (ax_it, ax_t):
        ax.axhline(a["rel_tol"], color="k", ls="--", lw=1)
    ax_acc.axhspan(*cmp["acc_range"], color="green", alpha=0.1)
    ax_it.set(title=f"N = {N}: |E − E_0| / E_0", xlabel="SR iteration")
    ax_t.set(title=f"N = {N}: |E − E_0| / E_0", xlabel="wall time [s]")
    ax_acc.set(title=f"N = {N}: acceptance", xlabel="SR iteration")
    ax_it.legend()
fig.tight_layout()
plt.show()

## 2. Efficiency against N

At the reference θ, recording every step:

- **ESS/s** (effective samples of $\sum_i|x_i|^2$ per second) is the figure of merit: higher is better.
- **τ per step** shows how correlated consecutive steps are. Metropolis moves all particles with a step that must shrink as N grows, so its τ is expected to grow with N. Gibbs moves each particle on its own and resamples the hidden units exactly.
- **Time per step**: a Gibbs step does the hidden units plus `n_metro` moves of every particle, so it is more expensive.
- **Agreement**: the difference between the two samplers in units of the combined error bar. Values inside the ±2σ band mean both sample the same $|\psi|^2$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
(ax_ess, ax_tau), (ax_time, ax_agree) = axes
for name in SAMPLER_NAMES:
    fx = [res[N]["fixed"][name] for N in Ns]
    c = SAMPLER_COLORS[name]
    ax_ess.plot(Ns, [f["r2"]["ess_per_s"] for f in fx], "o-", color=c, label=name)
    ax_tau.plot(Ns, [f["r2"]["tau"] for f in fx], "o-", color=c, label=name)
    ax_time.plot(Ns, [1e6 * f["time_per_step"] for f in fx], "o-", color=c, label=name)
for obs, marker in [("r2", "o"), ("E", "s")]:
    ax_agree.plot(Ns, [res[N]["fixed"]["agreement"][obs] for N in Ns], marker, ls="-",
                  label="Σ|x|²" if obs == "r2" else "E_loc")
ax_agree.axhspan(-2, 2, color="green", alpha=0.1)
ax_agree.axhline(0, color="k", lw=1)

ax_ess.set(title="Effective samples per second (Σ|x|²)", xlabel="N", yscale="log")
ax_tau.set(title="Autocorrelation time per step (Σ|x|²)", xlabel="N", yscale="log")
ax_time.set(title="Wall time per step [μs]", xlabel="N", yscale="log")
ax_agree.set(title="Metropolis − Gibbs [σ]", xlabel="N")
for ax in axes.flat:
    ax.legend()
fig.tight_layout()
plt.show()

for N in Ns:
    fx = res[N]["fixed"]
    gain = fx["gibbs"]["r2"]["ess_per_s"] / fx["metropolis"]["r2"]["ess_per_s"]
    winner = "Gibbs" if gain > 1 else "Metropolis"
    print(f"N = {N:>3}: {winner} gives {max(gain, 1 / gain):.1f}× more effective samples per second")